In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path
x_train = pd.read_csv(Path.cwd().parent / "data/german_credit_train.csv")
x_test = pd.read_csv(Path.cwd().parent / "data/german_credit_test.csv")


In [3]:
df.head()

,CheckingStatus,LoanDuration,CreditHistory,LoanPurpose,LoanAmount,ExistingSavings,EmploymentDuration,InstallmentPercent,Sex,OthersOnLoan,...,OwnsProperty,Age,InstallmentPlans,Housing,ExistingCreditsCount,Job,Dependents,Telephone,ForeignWorker,Risk
0,0_to_200,31,credits_paid_to_date,other,1889,100_to_500,less_1,3,female,none,...,savings_insurance,32,none,own,1,skilled,1,none,yes,No Risk
1,less_0,18,credits_paid_to_date,car_new,462,less_100,1_to_4,2,female,none,...,savings_insurance,37,stores,own,2,skilled,1,none,yes,No Risk
2,less_0,15,prior_payments_delayed,furniture,250,less_100,1_to_4,2,male,none,...,real_estate,28,none,own,2,skilled,1,yes,no,No Risk
3,0_to_200,28,credits_paid_to_date,retraining,3693,less_100,greater_7,3,male,none,...,savings_insurance,32,none,own,1,skilled,1,none,yes,No Risk
4,no_checking,28,prior_payments_delayed,education,6235,500_to_1000,greater_7,3,male,none,...,unknown,57,none,own,2,skilled,1,none,yes,Risk


In [2]:
df = x_train.copy()

In [9]:
y_sample = df["Risk"]
x_train = df.drop(columns="Risk", axis=1)

In [14]:
X_sample_num = x_train.select_dtypes(include = np.number)
X_sample_cat = x_train.select_dtypes(exclude = np.number)

ordinal_cols = X_sample_cat.columns
scale_cols = X_sample_num.columns

In [19]:
from sklearn.preprocessing import OrdinalEncoder, StandardScaler

scaler = StandardScaler()
ordinal = OrdinalEncoder()

x_train[scale_cols] = scaler.fit_transform(x_train[scale_cols])

x_train[ordinal_cols] = ordinal.fit_transform(x_train[ordinal_cols])

In [24]:
from sklearn.feature_selection import RFECV
from sklearn.model_selection import TimeSeriesSplit
from sklearn.ensemble import GradientBoostingClassifier

estimator = GradientBoostingClassifier(random_state=2512)

# Set the minimum number of features to be selected
min_features_to_select = 1

# Set the cross-validation splitting strategy
cv = TimeSeriesSplit(5)

# Create a RFECV object using the estimator
rfecv = RFECV(estimator, min_features_to_select=min_features_to_select, cv=cv)

# Fit the data
rfecv.fit(x_train, y_sample)

# Get integer index of the features selected
feature_index = rfecv.get_support(indices=True)

# Get a mask of the features selected
feature_mask = rfecv.support_

# Get selected feature names
feature_names = rfecv.get_feature_names_out()

# Get the number of features retained
feature_number = rfecv.n_features_

# Get results
results = pd.DataFrame(rfecv.cv_results_)

# Get RFECV score
rfecv_score = rfecv.score(x_train, y_sample)

# Print feature number, names and score
print("Original feature number:", len(x_train.columns))
print("Optimal feature number:", feature_number)
print("Selected features:", feature_names)
print("Score:", rfecv_score)

Original feature number: 20
Optimal feature number: 16
Selected features: ['CheckingStatus' 'LoanDuration' 'CreditHistory' 'LoanAmount'
 'ExistingSavings' 'EmploymentDuration' 'InstallmentPercent' 'Sex'
 'OthersOnLoan' 'CurrentResidenceDuration' 'OwnsProperty' 'Age'
 'InstallmentPlans' 'ExistingCreditsCount' 'Job' 'Telephone']
Score: 0.835458864716179
